# Loading Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
from scipy.stats import zscore

In [2]:
df = pd.read_csv('Dataset/df_clean_eng.csv')
df.head()

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,cad,appet,pe,ane,classification,eGFR,comorb_score,anemia_severity,kidney_func_score,symptom_severity
0,48.0,80.0,1.020,1,0,missing,normal,notpresent,notpresent,121.000000,...,no,good,no,no,ckd,68.682456,2,-2.384153,-0.788205,0
1,7.0,50.0,1.020,4,0,missing,normal,notpresent,notpresent,140.798289,...,no,good,no,no,ckd,162.103427,0,0.807377,-1.296071,0
2,62.0,80.0,1.010,2,3,normal,normal,notpresent,notpresent,423.000000,...,no,poor,no,yes,ckd,40.838799,1,2.907140,-0.252869,2
3,48.0,70.0,1.005,4,0,normal,abnormal,present,notpresent,117.000000,...,no,poor,yes,yes,ckd,18.161458,1,2.028862,4.596228,3
4,51.0,80.0,1.010,2,0,normal,normal,notpresent,notpresent,106.000000,...,no,good,no,no,ckd,56.786414,0,0.759758,-0.964553,0


In [3]:
# Converting Object to Category
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype('category')

Ensuring the datatypes and non-null values are fine before moving on to train-test-split, scaling and transforms

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 396 entries, 0 to 395
Data columns (total 30 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   age                396 non-null    float64 
 1   bp                 396 non-null    float64 
 2   sg                 396 non-null    float64 
 3   al                 396 non-null    int64   
 4   su                 396 non-null    int64   
 5   rbc                396 non-null    category
 6   pc                 396 non-null    category
 7   pcc                396 non-null    category
 8   ba                 396 non-null    category
 9   bgr                396 non-null    float64 
 10  bu                 396 non-null    float64 
 11  sc                 396 non-null    float64 
 12  sod                396 non-null    float64 
 13  pot                396 non-null    float64 
 14  hemo               396 non-null    float64 
 15  pcv                396 non-null    float64 
 16  wc      

# Train & Test Splitting

## Transformation and Scaling Plan

> Data Transformation Steps:
> - Log-transformed highly skewed variables: bgr, bu, sc, wc, eGFR.
> - Standard scaled all numerical features.
> - One-hot encoded all categorical variables.
> - All transformations were fit on the training set and applied to the test set to prevent data leakage.

# Version 0

Ensuring None That Require Log Transforms Have 0 or -ve Values

In [55]:
cols_to_check = ['bgr', 'bu', 'sc', 'wc', 'eGFR']

for col in cols_to_check:
    if col in df.columns:
        num_zeros = (df[col] == 0).sum()
        num_neg = (df[col] < 0).sum()
        print(f"{col}: {num_zeros} zeros, {num_neg} negatives")
        print(f"Min value: {df[col].min()}")
        print(f"Type: {df[col].dtype}\n")

bgr: 0 zeros, 0 negatives
Min value: 22.0
Type: float64

bu: 0 zeros, 0 negatives
Min value: 1.5
Type: float64

sc: 0 zeros, 0 negatives
Min value: 0.149489444642489
Type: float64

wc: 0 zeros, 0 negatives
Min value: 2200.0
Type: float64

eGFR: 0 zeros, 0 negatives
Min value: 0.9016358277259284
Type: float64



In [56]:

target = 'classification'
feature_cols = [col for col in df.columns if col not in [target, 'sc_bin', 'hemo_bin', 'bu_bin']]  # Exclude binned cols and target

X = df[feature_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [57]:
print(X_train.columns.tolist())

['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'eGFR', 'comorb_score', 'anemia_severity', 'kidney_func_score', 'symptom_severity']


In [58]:
print(X_train.head())

      age    bp        sg  al  su       rbc        pc         pcc          ba  \
232  45.0  70.0  1.010000   2   0   missing    normal  notpresent  notpresent   
145  69.0  60.0  1.017386   1   0   missing   missing  notpresent  notpresent   
239  69.0  70.0  1.010000   4   3    normal  abnormal     present     present   
290  75.0  60.0  1.020000   0   0    normal    normal  notpresent  notpresent   
142  57.0  90.0  1.015000   5   0  abnormal  abnormal  notpresent     present   

            bgr  ...   dm  cad  appet   pe  ane        eGFR  comorb_score  \
232  113.000000  ...   no  yes   good   no  yes   32.845576             1   
145  171.000000  ...   no   no   poor   no   no    0.901636             1   
239  214.000000  ...  yes  yes   good  yes  yes    9.414290             3   
290  110.000000  ...   no   no   good   no   no  116.850983             0   
142  195.313575  ...  yes  yes   poor  yes  yes    4.242088             3   

     anemia_severity kidney_func_score symptom_sev

In [59]:
print(y)

0         ckd
1         ckd
2         ckd
3         ckd
4         ckd
        ...  
391    notckd
392    notckd
393    notckd
394    notckd
395    notckd
Name: classification, Length: 396, dtype: category
Categories (2, object): ['ckd', 'notckd']


In [60]:
log_transform_cols = ['bgr', 'bu', 'sc', 'wc', 'eGFR']

for col in log_transform_cols:
    for x in [X_train, X_test]:
        x[col] = np.log(x[col])

In [61]:
X_train.head()

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,dm,cad,appet,pe,ane,eGFR,comorb_score,anemia_severity,kidney_func_score,symptom_severity
232,45.0,70.0,1.010000,2,0,missing,normal,notpresent,notpresent,4.727388,...,no,yes,good,no,yes,3.491817,1,4.691069,0.778275,1
145,69.0,60.0,1.017386,1,0,missing,missing,notpresent,notpresent,5.141664,...,no,no,poor,no,no,-0.103545,1,0.044029,10.737466,1
239,69.0,70.0,1.010000,4,3,normal,abnormal,present,present,5.365976,...,yes,yes,good,yes,yes,2.242229,3,3.829682,4.676520,2
290,75.0,60.0,1.020000,0,0,normal,normal,notpresent,notpresent,4.700480,...,no,no,good,no,no,4.760899,0,-2.147557,-0.188470,0
142,57.0,90.0,1.015000,5,0,abnormal,abnormal,notpresent,present,5.274606,...,yes,yes,poor,yes,yes,1.445056,3,4.829802,10.505472,3


In [62]:
# Identify numerical columns (excluding categorical and binned)
num_cols = X_train.select_dtypes(include=['float64', 'int32']).columns.tolist()

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [63]:
X_train.head()

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,dm,cad,appet,pe,ane,eGFR,comorb_score,anemia_severity,kidney_func_score,symptom_severity
232,-0.364216,-0.441821,-1.444439,2,0,missing,normal,notpresent,notpresent,-0.422890,...,no,yes,good,no,yes,-0.314845,1,1.661909,0.299056,1
145,1.019392,-1.164782,-0.053023,1,0,missing,missing,notpresent,notpresent,0.559469,...,no,no,poor,no,no,-3.746295,1,0.028827,4.357767,1
239,1.019392,-0.441821,-1.444439,4,3,normal,abnormal,present,present,1.091373,...,yes,yes,good,yes,yes,-1.507465,3,1.359197,1.887724,2
290,1.365294,-1.164782,0.439451,0,0,normal,normal,notpresent,notpresent,-0.486695,...,no,no,good,no,no,0.896381,0,-0.741349,-0.094926,0
142,0.327588,1.004101,-0.502494,5,0,abnormal,abnormal,notpresent,present,0.874711,...,yes,yes,poor,yes,yes,-2.268295,3,1.710663,4.263221,3


In [64]:
cat_cols = X_train.select_dtypes(include='category').columns.tolist()

X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)

# Ensure columns match in train and test
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

In [65]:
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

print(le.classes_)  # To see which label is 0 and which is 1

y_train = pd.Series(y_train)
y_test = pd.Series(y_test)

['ckd' 'notckd']


In [66]:
X_train.head()

,age,bp,sg,al,su,bgr,bu,sc,sod,pot,...,pc_missing,pc_normal,pcc_present,ba_present,htn_yes,dm_yes,cad_yes,appet_poor,pe_yes,ane_yes
232,-0.364216,-0.441821,-1.444439,2,0,-0.422890,1.136664,0.330391,-0.000938,0.306884,...,False,True,False,False,False,False,True,False,False,True
145,1.019392,-1.164782,-0.053023,1,0,0.559469,-0.789703,3.767945,-0.000938,0.306884,...,True,False,False,False,True,False,False,True,False,False
239,1.019392,-0.441821,-1.444439,4,3,1.091373,1.184651,1.469665,-2.779998,-0.799084,...,False,False,True,True,True,True,True,False,True,True
290,1.365294,-1.164782,0.439451,0,0,-0.486695,0.198684,-1.014595,-0.401854,0.902794,...,False,True,False,False,False,False,False,False,False,False
142,0.327588,1.004101,-0.502494,5,0,0.874711,3.013832,2.288697,-1.828740,0.593361,...,False,False,False,True,True,True,True,True,True,True


In [67]:
print(X_train.isnull().sum().sum(), X_test.isnull().sum().sum())

0 0


In [68]:
print(X_train.shape, X_test.shape)

(316, 31) (80, 31)


In [69]:
print(df.columns.tolist())

['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'classification', 'eGFR', 'comorb_score', 'anemia_severity', 'kidney_func_score', 'symptom_severity']


# Version 1

In [22]:

feature_cols_imp1 = ['anemia_severity','hemo','sg','comorb_score','eGFR','rbc','sc','dm','htn','bgr','symptom_severity']

X_imp1 = df[feature_cols_imp1]
y_imp1 = df[target]

X_train_imp1, X_test_imp1, y_train_imp1, y_test_imp1 = train_test_split(X_imp1, y_imp1, test_size=0.2, random_state=42, stratify=y_imp1)

In [23]:
y_train_imp1.value_counts()

classification
ckd       196
notckd    120
Name: count, dtype: int64

In [24]:
len(feature_cols_imp1)

11

In [25]:
log_transform_cols_imp1 = ['bgr','sc', 'eGFR']

for col in log_transform_cols_imp1:
    for x in [X_train_imp1, X_test_imp1]:
        x[col] = np.log(x[col])

In [26]:
# Identify numerical columns (excluding categorical and binned)
num_cols_imp1 = X_train_imp1.select_dtypes(include=['float64', 'int32']).columns.tolist()

scaler_imp1 = StandardScaler()
X_train_imp1[num_cols_imp1] = scaler_imp1.fit_transform(X_train_imp1[num_cols_imp1])
X_test_imp1[num_cols_imp1] = scaler_imp1.transform(X_test_imp1[num_cols_imp1])

In [27]:
cat_cols_imp1 = X_train_imp1.select_dtypes(include='category').columns.tolist()

X_train_imp1 = pd.get_dummies(X_train_imp1, columns=cat_cols_imp1, drop_first=True)
X_test_imp1 = pd.get_dummies(X_test_imp1, columns=cat_cols_imp1, drop_first=True)

# Ensure columns match in train and test
X_test_imp1 = X_test_imp1.reindex(columns=X_train_imp1.columns, fill_value=0)

In [28]:
y_test_imp1.value_counts()

classification
ckd       50
notckd    30
Name: count, dtype: int64

In [29]:
type(y_train_imp1)

pandas.core.series.Series

In [30]:
le_imp1 = LabelEncoder()
y_train_imp1 = le_imp1.fit_transform(y_train_imp1)
y_test_imp1 = le_imp1.transform(y_test_imp1)

print(le_imp1.classes_)  
y_train_imp1 = pd.Series(y_train_imp1)
y_test_imp1 = pd.Series(y_test_imp1)

['ckd' 'notckd']


In [31]:
print(y_train_imp1.value_counts())
print(y_test_imp1.value_counts())

0    196
1    120
Name: count, dtype: int64
0    50
1    30
Name: count, dtype: int64


# Version 2

In [32]:
# Exclude target and categorical columns from features
feature_cols_imp2 = ['hemo','sg','sc','htn','bgr']

X_imp2 = df[feature_cols_imp2]
y_imp2 = df[target]

X_train_imp2, X_test_imp2, y_train_imp2, y_test_imp2 = train_test_split(X_imp2, y_imp2, test_size=0.2, random_state=42, stratify=y_imp2)

In [33]:
y_test_imp2.value_counts()

classification
ckd       50
notckd    30
Name: count, dtype: int64

In [34]:
len(feature_cols_imp2)

5

In [35]:
log_transform_cols_imp2 = ['bgr','sc']

for col in log_transform_cols_imp2:
    for x in [X_train_imp2, X_test_imp2]:
        x[col] = np.log(x[col])

In [36]:
# Identify numerical columns (excluding categorical and binned)
num_cols_imp2 = X_train_imp2.select_dtypes(include=['float64', 'int32']).columns.tolist()

scaler_imp2 = StandardScaler()
X_train_imp2[num_cols_imp2] = scaler_imp2.fit_transform(X_train_imp2[num_cols_imp2])
X_test_imp2[num_cols_imp2] = scaler_imp2.transform(X_test_imp2[num_cols_imp2])

In [37]:
cat_cols_imp2 = X_train_imp2.select_dtypes(include='category').columns.tolist()

X_train_imp2 = pd.get_dummies(X_train_imp2, columns=cat_cols_imp2, drop_first=True)
X_test_imp2 = pd.get_dummies(X_test_imp2, columns=cat_cols_imp2, drop_first=True)

# Ensure columns match in train and test
X_test_imp2 = X_test_imp2.reindex(columns=X_train_imp2.columns, fill_value=0)

In [38]:
le_imp2 = LabelEncoder()
y_train_imp2 = le_imp2.fit_transform(y_train_imp2)
y_test_imp2 = le_imp2.transform(y_test_imp2)

print(le_imp2.classes_) 

y_train_imp2 = pd.Series(y_train_imp2)
y_test_imp2 = pd.Series(y_test_imp2)

['ckd' 'notckd']


In [39]:
X_test_imp2.isnull().sum().sum()

np.int64(0)